In [1]:
%load_ext autoreload
%autoreload 2

# Work risk

In [3]:
from fuzzyroughrules.rule_induction_base import RuleGenerator
from fuzzyroughrules.approximations import LowerApproximation
from fuzzyroughrules.operators import ImplicatorInclusion
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

In [4]:
data = pd.read_csv(Path.cwd() / 'other-data' / 'full-dataset.csv', sep=",")
data

,worker ID,Body part,age,hight,weight,work experince,medical record,Station risk,shift,station,final risk,corrective action,problem,operation,environment,job frequency
0,1,Back,31,175,58,8,0,0.69,1,1,7.75,Hamstring & lumbar stretches,low back pain/sciatica risk,42,49,75
1,1,neck,31,175,58,8,0,0.63,1,1,7.75,Neck side bends,Neck strain,50,38,23
2,1,hand,31,175,58,8,0,0.48,1,1,3.00,wall angels,shoulder tendonitis,41,20,33
3,1,knee,31,175,58,8,0,0.44,1,1,5.00,mini squates,Prolonged standing,15,43,60
4,2,back,32,180,80,8,1,0.69,2,1,8.00,Cat-cow stretch/glute bridge,low back pain/sciatica risk,42,49,75
5,2,neck,32,180,80,8,0,0.63,2,1,7.80,Neck side bends,Neck strain,50,38,23
6,2,hand,32,180,80,8,0,0.48,2,1,3.00,nerve gliding,shoulder tendonitis,41,20,33
7,2,knee,32,180,80,8,0,0.44,2,1,5.20,mini squates,Prolonged standing,15,43,60
8,3,back,23,187,95,1,1,0.73,1,2,6.50,seated spinal twist,Low back pain,55,40,65
9,3,neck,23,187,95,1,0,0.60,1,2,5.50,Chin tucks,Cervicalgia,55,39,33


In [10]:
data.dtypes

worker ID              int64
Body part             object
age                    int64
hight                  int64
weight                 int64
work experince         int64
medical record         int64
Station risk         float64
shift                  int64
station                int64
final risk           float64
corrective action     object
problem               object
operation              int64
environment            int64
job frequency          int64
dtype: object

In [12]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in data.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

In [13]:
data

,worker ID,Body part,age,hight,weight,work experince,medical record,Station risk,shift,station,final risk,corrective action,problem,operation,environment,job frequency
0,1,0,31,175,58,8,0,0.69,1,1,7.75,9,17,42,49,75
1,1,5,31,175,58,8,0,0.63,1,1,7.75,12,9,50,38,23
2,1,3,31,175,58,8,0,0.48,1,1,3.00,25,18,41,20,33
3,1,4,31,175,58,8,0,0.44,1,1,5.00,18,10,15,43,60
4,2,1,32,180,80,8,1,0.69,2,1,8.00,3,17,42,49,75
5,2,5,32,180,80,8,0,0.63,2,1,7.80,12,9,50,38,23
6,2,3,32,180,80,8,0,0.48,2,1,3.00,19,18,41,20,33
7,2,4,32,180,80,8,0,0.44,2,1,5.20,18,10,15,43,60
8,3,1,23,187,95,1,1,0.73,1,2,6.50,20,7,55,40,65
9,3,5,23,187,95,1,0,0.60,1,2,5.50,5,0,55,39,33


In [52]:
label_encoders['Body part'].transform(['neck'])

array([5])

In [59]:
test_obj = [5, 28, 186, 78, 3, 1, 1, 1, 34, 20, 20]

In [60]:
x = data.drop(['worker ID', 'corrective action', 'problem', 'Station risk', 'final risk'], axis=1).to_numpy()
y = data['corrective action'].to_numpy()
x, y

(array([[  0,  31, 175,  58,   8,   0,   1,   1,  42,  49,  75],
        [  5,  31, 175,  58,   8,   0,   1,   1,  50,  38,  23],
        [  3,  31, 175,  58,   8,   0,   1,   1,  41,  20,  33],
        [  4,  31, 175,  58,   8,   0,   1,   1,  15,  43,  60],
        [  1,  32, 180,  80,   8,   1,   2,   1,  42,  49,  75],
        [  5,  32, 180,  80,   8,   0,   2,   1,  50,  38,  23],
        [  3,  32, 180,  80,   8,   0,   2,   1,  41,  20,  33],
        [  4,  32, 180,  80,   8,   0,   2,   1,  15,  43,  60],
        [  1,  23, 187,  95,   1,   1,   1,   2,  55,  40,  65],
        [  5,  23, 187,  95,   1,   0,   1,   2,  55,  39,  33],
        [  3,  23, 187,  95,   1,   0,   1,   2,  30,  25,  33],
        [  4,  23, 187,  95,   1,   0,   1,   2,   5,  34,  80],
        [  1,  25, 168,  65,   1,   1,   2,   2,  55,  40,  65],
        [  5,  25, 168,  65,   1,   0,   2,   2,  55,  39,  33],
        [  3,  25, 168,  65,   1,   0,   2,   2,  30,  25,  33],
        [  4,  25, 168,  

In [61]:
scaler = MinMaxScaler()
scaler.fit(x)
x_scaled = scaler.transform(x)
x_scaled

array([[0.00000, 0.34783, 0.40000, 0.07500, 0.38889, 0.00000, 0.00000,
        0.00000, 0.41111, 0.67442, 0.90909],
       [1.00000, 0.34783, 0.40000, 0.07500, 0.38889, 0.00000, 0.00000,
        0.00000, 0.50000, 0.41860, 0.23377],
       [0.60000, 0.34783, 0.40000, 0.07500, 0.38889, 0.00000, 0.00000,
        0.00000, 0.40000, 0.00000, 0.36364],
       [0.80000, 0.34783, 0.40000, 0.07500, 0.38889, 0.00000, 0.00000,
        0.00000, 0.11111, 0.53488, 0.71429],
       [0.20000, 0.39130, 0.65000, 0.62500, 0.38889, 1.00000, 1.00000,
        0.00000, 0.41111, 0.67442, 0.90909],
       [1.00000, 0.39130, 0.65000, 0.62500, 0.38889, 0.00000, 1.00000,
        0.00000, 0.50000, 0.41860, 0.23377],
       [0.60000, 0.39130, 0.65000, 0.62500, 0.38889, 0.00000, 1.00000,
        0.00000, 0.40000, 0.00000, 0.36364],
       [0.80000, 0.39130, 0.65000, 0.62500, 0.38889, 0.00000, 1.00000,
        0.00000, 0.11111, 0.53488, 0.71429],
       [0.20000, 0.00000, 1.00000, 1.00000, 0.00000, 1.00000, 0.00000,
 

In [62]:
model = RuleGenerator(
    with_reducts=True
)

In [63]:
model.fit(x, y, _)

RuleGenerator(with_reducts=True, apply_relabelling=False, relabelling_threshold=0.0, discard_uncertain_objects=False, add_uncovered_objects=True, certainty_threshold=0.0, print_nr_of_rule_candidates=False, print_changes=False, optimise_attribute_order=False, optimise_slopes=False, slope_options=None, covering_threshold=1e-06, inclusion_threshold=0.999999, priors_influence=0, approximation=None, inclusion_measure=None, attribute_ordering=None)

In [64]:
model.get_rules()

array([Rule(antecedent=array([0.00000, 0.34783, 0.40000, 0.07500, 0.38889, 0.00000, 0.00000,
              0.00000, 0.41111, 0.67442, 0.90909]), reducts=array([RelationTypes.DOMINANT, RelationTypes.UNUSED, RelationTypes.UNUSED,
              RelationTypes.DOMINANT, RelationTypes.UNUSED, RelationTypes.UNUSED,
              RelationTypes.DOMINANT, RelationTypes.DOMINANT,
              RelationTypes.UNUSED, RelationTypes.UNUSED,
              RelationTypes.DOMINATED], dtype=object), slopes=array([1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000,
              1.00000, 1.00000, 1.00000, 1.00000]), credibility=0.6000000000000001, decision=9)                                       ,
       Rule(antecedent=array([1.00000, 0.34783, 0.40000, 0.07500, 0.38889, 0.00000, 0.00000,
              0.00000, 0.50000, 0.41860, 0.23377]), reducts=array([RelationTypes.UNUSED, RelationTypes.UNUSED, RelationTypes.UNUSED,
              RelationTypes.DOMINANT, RelationTypes.UNUSED, RelationTypes.UN

In [57]:
model.predict([test_obj])

array([20])

In [58]:
label_encoders['corrective action'].inverse_transform([model.predict([test_obj])
])

/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:155: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


array(['seated spinal twist'], dtype=object)

In [39]:
x.shape

(52, 11)

In [32]:
from fuzzyroughrules.full_inclusion_frri import GranularRuleGenerator
from fuzzyroughrules.rule_induction_base import RuleGenerator
from fuzzyroughrules.approximations import MulticlassMSECVXApproximation
from fuzzyroughrules.operators import StrictFuzzyInclusion
from pathlib import Path

gran_model = RuleGenerator(
    approximation=MulticlassMSECVXApproximation(),
    inclusion_measure=StrictFuzzyInclusion(),
    apply_relabelling=True,
    covering_threshold=.50 - 1e-6,
)

In [33]:
from hhelper.data_loader import get_dataset

x, y = get_dataset(Path.cwd() / 'keel-data' / 'ecoli-10-fold', '1tra')
x_test, y_test = get_dataset(Path.cwd() / 'keel-data' / 'ecoli-10-fold', '1tst')

In [37]:
gran_model.fit(x, y, _)

1
[128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197]
[0.85132 0.41513 0.57938 0.75611 0.66450 0.69111 0.68648 0.61091 0.55461
 0.55114 0.74143 0.55161 0.45678 0.69747 0.62644 0.61425 0.52949 0.62055
 0.67888 0.57961 0.61530 0.64234 0.63932 0.76340 0.50518 0.78810 0.65351
 0.61159 0.54362 0.57778 0.57659 0.51321 0.45190 0.59110 0.78538 0.80524
 0.35978 0.62985 0.69747 0.50041 0.69747 0.52213 0.58020 0.59902 0.67549
 0.58427 0.54598 0.52165 0.52024 0.73044 0.67549 0.53890 0.59857 0.63154
 0.62055 0.63483 0.62569 0.65585 0.63154 0.55531 0.54408 0.63777 0.62128
 0.70846 0.57659 0.57659 0.51066 0.51310 0.62877 0.63946]
2
[199, 200]
[0.99992 0.71264]
3
[198]
[0.58683]
4
[201, 202, 203, 204, 205, 2

GranularRuleGenerator(with_reducts=True, apply_relabelling=True, relabelling_threshold=0.0, discard_uncertain_objects=False, add_uncovered_objects=True, certainty_threshold=0.0, print_nr_of_rule_candidates=False, print_changes=False, optimise_attribute_order=False, optimise_slopes=False, slope_options=None, covering_threshold=0.499999, inclusion_threshold=0.999999, priors_influence=0, use_gran_approx=False, approximation=MulticlassMSECVXApproximation(weights=None, nn_approx=-1, n_jobs=None), inclusion_measure=<fuzzyroughrules.operators.StrictFuzzyInclusion object at 0x31d230340>, attribute_ordering=None)

In [38]:
gran_model.predict(x_test)

array([' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' im', ' cp', ' im',
       ' im', ' im', ' im', ' im', ' im', ' im', ' im', ' im', ' pp',
       ' cp', ' omL', ' cp', ' cp', ' pp', ' pp', ' pp'], dtype=object)

In [39]:
model2.get_rules_as_string()

["[{'attribute': 2, 'type': 'l', 'bounds': [66.0]}, {'attribute': 7, 'type': 'l', 'bounds': [36.4]}], class: 1",
 "[{'attribute': 0, 'type': 'r', 'bounds': [7.565217391304348]}, {'attribute': 1, 'type': 'r', 'bounds': [187.0]}], class: 0"]

In [23]:
model2.selected_indexes

array([1, 4])

In [37]:
from fuzzyroughrules.approximations import LowerApproximation

appr = LowerApproximation()
pos_reg = appr.get_approximation(x_scaled, y)

In [38]:
pos_reg

array([0.93478, 0.60000, 0.58809, 0.58809, 0.94565, 0.89130, 0.91667])

Stap per stap berekening van appr.get_approximation(x_scaled, y)

In [42]:
from fuzzyroughrules.fuzzy_operators import triangular_similarity, discernibility_matrix, get_ind_low_apr, lukasiewicz_implicator

rel_mat = triangular_similarity(x_scaled, x_scaled)

In [43]:
rel_mat

array([[1.00000, 0.00000, 0.00000, 0.06522, 0.04255, 0.36397, 0.60000],
       [0.00000, 1.00000, 0.40000, 0.45000, 0.00000, 0.00000, 0.08333],
       [0.00000, 0.40000, 1.00000, 0.41191, 0.12500, 0.12500, 0.00654],
       [0.06522, 0.45000, 0.41191, 1.00000, 0.05435, 0.10870, 0.00000],
       [0.04255, 0.00000, 0.12500, 0.05435, 1.00000, 0.00000, 0.00000],
       [0.36397, 0.00000, 0.12500, 0.10870, 0.00000, 1.00000, 0.19485],
       [0.60000, 0.08333, 0.00654, 0.00000, 0.00000, 0.19485, 1.00000]])

In [44]:
fuzzy_set = discernibility_matrix(y, y)

In [45]:
fuzzy_set

array([[1, 0, 1, 0, 1, 1, 1],
       [0, 1, 0, 1, 0, 0, 0],
       [1, 0, 1, 0, 1, 1, 1],
       [0, 1, 0, 1, 0, 0, 0],
       [1, 0, 1, 0, 1, 1, 1],
       [1, 0, 1, 0, 1, 1, 1],
       [1, 0, 1, 0, 1, 1, 1]])

In [47]:
impl = lukasiewicz_implicator(rel_mat, fuzzy_set)
impl

array([[1.00000, 1.00000, 1.00000, 0.93478, 1.00000, 1.00000, 1.00000],
       [1.00000, 1.00000, 0.60000, 1.00000, 1.00000, 1.00000, 0.91667],
       [1.00000, 0.60000, 1.00000, 0.58809, 1.00000, 1.00000, 1.00000],
       [0.93478, 1.00000, 0.58809, 1.00000, 0.94565, 0.89130, 1.00000],
       [1.00000, 1.00000, 1.00000, 0.94565, 1.00000, 1.00000, 1.00000],
       [1.00000, 1.00000, 1.00000, 0.89130, 1.00000, 1.00000, 1.00000],
       [1.00000, 0.91667, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000]])

In [51]:
np.min(impl, 0)

array([0.93478, 0.60000, 0.58809, 0.58809, 0.94565, 0.89130, 0.91667])

Voor sectie 3.3 Rule selection

In [82]:
from fuzzyroughrules.fuzzy_operators import lukasiewicz_t_norm
from fuzzyroughrules.operators import triangular_relation

covering = np.zeros((model.n_samples, model.n_samples))
for i in range(model.n_samples):
    current_covering = lukasiewicz_t_norm(
        # fo.general_triangular_relation(self.X_scaled, self.X_scaled[i], self.theta, self.reducts[i]),
        triangular_relation(
            model.X_scaled,
            model.X_scaled[i],
            model.slopes[i],
            model.reducts[i]
        ),
        model.positive_region[i]
    )
    if len(current_covering.shape) > 1:
        current_covering = current_covering.T
    covering[i, :] = (1 * (current_covering > model.covering_threshold))

In [83]:
model.covering_threshold

1e-06

In [84]:
covering

array([[1.00000, 0.00000, 1.00000, 0.00000, 0.00000, 1.00000, 1.00000],
       [0.00000, 1.00000, 0.00000, 1.00000, 0.00000, 0.00000, 0.00000],
       [1.00000, 0.00000, 1.00000, 0.00000, 1.00000, 0.00000, 1.00000],
       [0.00000, 1.00000, 0.00000, 1.00000, 0.00000, 0.00000, 0.00000],
       [1.00000, 0.00000, 1.00000, 0.00000, 1.00000, 1.00000, 1.00000],
       [1.00000, 0.00000, 1.00000, 0.00000, 1.00000, 1.00000, 1.00000],
       [1.00000, 0.00000, 1.00000, 0.00000, 1.00000, 1.00000, 1.00000]])

In [81]:
from fuzzyroughrules.operators import RelationTypes

lukasiewicz_t_norm(
    triangular_relation(
        x_scaled[3],
        x_scaled[0], 
        slopes=np.ones(model.n_attributes, dtype=float),
        rel_types=np.array([RelationTypes.UNUSED, RelationTypes.DOMINANT, RelationTypes.INDISCERNIBLE, RelationTypes.INDISCERNIBLE, RelationTypes.INDISCERNIBLE, RelationTypes.INDISCERNIBLE, RelationTypes.INDISCERNIBLE, RelationTypes.INDISCERNIBLE])
    ),
    model.positive_region[0]
)


array([0.00000])

In [80]:
model.reducts

array([[<RelationTypes.UNUSED: 0>, <RelationTypes.DOMINANT: -1>,
        <RelationTypes.DOMINANT: -1>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.UNUSED: 0>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.UNUSED: 0>, <RelationTypes.UNUSED: 0>],
       [<RelationTypes.UNUSED: 0>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.DOMINATED: 1>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.UNUSED: 0>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.UNUSED: 0>, <RelationTypes.DOMINATED: 1>],
       [<RelationTypes.UNUSED: 0>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.DOMINANT: -1>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.UNUSED: 0>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.DOMINATED: 1>, <RelationTypes.UNUSED: 0>],
       [<RelationTypes.UNUSED: 0>, <RelationTypes.DOMINATED: 1>,
        <RelationTypes.UNUSED: 0>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.UNUSED: 0>, <RelationTypes.UNUSED: 0>,
        <RelationTypes.DOMINANT: -1>, <Relatio

In [74]:
model.positive_region[0]

0.9347826086956521

In [58]:
model.inclusion_threshold

0.9999

In [62]:
model2.reducts

array([[ 0, -1, -1,  0,  0,  0,  0,  0],
       [ 0,  0,  1,  0,  0,  0, -1,  0],
       [ 0,  0, -1,  0,  0,  0,  1,  0],
       [ 0,  1,  0,  0,  0,  0, -1,  0],
       [-1, -1,  0,  0,  0,  0,  0,  0],
       [ 0, -1,  0,  0,  0,  0,  0, -1],
       [ 0, -1,  0,  0,  0,  0,  0, -1]])